In [1]:
import os
import json
import yaml
from typing import Iterable, List, Union

Find checkpoints with the given conditions

In [2]:
def matches_conditions(config_data, conditions):
    """Recursively check if config_data satisfies the given conditions."""
    for key, expected_value in conditions.items():
        if key not in config_data:
            return False

        actual_value = config_data[key]
        if isinstance(expected_value, dict):
            if not isinstance(actual_value, dict):
                return False
            if not matches_conditions(actual_value, expected_value):
                return False
        elif actual_value != expected_value:
            return False
    return True


def load_config_file(config_path: str):
    """Load configuration based on file extension. Supports .json / .yaml / .yml."""
    extension = os.path.splitext(config_path)[1].lower()
    with open(config_path, "r", encoding="utf-8") as config_file:
        if extension == ".json":
            return json.load(config_file)
        if extension in (".yaml", ".yml"):
            return yaml.safe_load(config_file)
        raise ValueError(f"Unsupported configuration format: {extension}")


def find_configs_by_conditions(
    conditions: dict,
    root_dir: str,
    filenames: Union[str, Iterable[str]] = ("config.yaml", "config.json"),
) -> List[str]:
    """Recursively find directories whose config file matches the given conditions."""
    if isinstance(filenames, str):
        target_filenames = {filenames}
    else:
        target_filenames = set(filenames)

    matched_directories = []

    for current_dir, _, files in os.walk(root_dir):
        matching_filenames = target_filenames.intersection(files)
        for filename in matching_filenames:
            config_path = os.path.join(current_dir, filename)
            try:
                config_data = load_config_file(config_path)
                if isinstance(config_data, dict) and matches_conditions(config_data, conditions):
                    matched_directories.append(os.path.abspath(current_dir))
                    break
            except Exception as exc:
                print(f"[Error] Failed to read {config_path}: {exc}")

    return matched_directories


In [3]:
checkpoints_root_dir = "./checkpoints"

condition_clf_mixer_qtm_salton_sea = {
    # "load_specific_parts": ["encoder"],
    "model": "clf_mixer_attnpl_t",
    "mixer_model_config": {
        # "d_model": 64,
        # "attn_layer_idx": [],
        # "ssm_cfg": {"layer": "Mamba2"}
    },
    "dataset": "QTMSaltonSea",
    # "Mf": 4.5,
    # "Twindow": 180,
    # "Tfore": 20
}

condition_etas_pnr_1z = {
    "model": "rtpp",
    # "mixer_model_config": {
    #     # "d_model": 64,
    #     # "attn_layer_idx": [0],
    #     # "ssm_cfg": {"layer": "Mamba2"}
    # },
    # "predict_b": False,
    # "features_input_keys": ["mag"],  # "log_inter_times"
    "bg_model": "conv_mlp",
    "dataset": "PNR_1z",
}

condition_rtpp_chuandian = {
    "model": "rtpp",
    # "mixer_model_config": {
    #     # "d_model": 64,
    #     # "attn_layer_idx": [],
    #     # "ssm_cfg": {"layer": "Mamba2"}
    # },
    # "dMag": 0.1,
    "dataset": "ChuanDian",
}

condition_lstm_chuandian = {
    "model": "lstm",
    "dataset": "ChuanDian",
}

condition_reg_mixer_chuandian = {
    "model": "reg_mixer_attnpl_t",
    "dataset": "ChuanDian",
    "Twindow": 600,
}

condition_rf_grid = {
    # "model": "rf",
    "Mc": 0.6,
    "Mf": 3.7,
    "Twindow": 180,
    "Tfore": 60,
    "dt": 5,
    "context_len": 1,
    # "dataset": "ChuanDian",
    # "Mag_elaps": "[5, 5.5, 6, 6.5]"
}

condition1 = {
"model": "rtpp_v2",
"bg_model": "mamba",
"b_use_bg_context": False,
 "dataset": "St1-2018",
# "catalog_cfg": {
#     "mag_completeness": -0.5
#     }
}

matching_directories = find_configs_by_conditions(condition1, checkpoints_root_dir)

[Error] Failed to read ./checkpoints/lstm_20260130-213729/config.yaml: while scanning a simple key
  in "./checkpoints/lstm_20260130-213729/config.yaml", line 17, column 1
could not find expected ':'
  in "./checkpoints/lstm_20260130-213729/config.yaml", line 19, column 1


In [4]:
matching_directories
# matching_directories = ["checkpoints/rf_847e64da",
#                         "checkpoints/rf_dfae9b13",
#                         "checkpoints/rf_ba2359e6",
#                         "checkpoints/rf_9ffe46be",
#                         "checkpoints/rf_af684ff4"]


['/root/autodl-tmp/em_eqf/checkpoints/rtpp_v2_20260418-175918',
 '/root/autodl-tmp/em_eqf/checkpoints/rtpp_v2_20260418-181941',
 '/root/autodl-tmp/em_eqf/checkpoints/rtpp_v2_20260420-102834']

In [5]:
def load_metrics_from_directories(directory_list, metrics_filename):
    """Load metrics JSON files from directories if the target file exists."""
    metrics_records = []

    for directory_path in directory_list:
        metrics_file_path = os.path.join(directory_path, metrics_filename)
        if os.path.isfile(metrics_file_path):
            try:
                with open(metrics_file_path, "r", encoding="utf-8") as metrics_file:
                    metrics_data = json.load(metrics_file)

                metrics_records.append(
                    {
                        "directory_path": os.path.abspath(directory_path),
                        "metrics_data": metrics_data,
                    }
                )
            except Exception as exc:
                print(f"An error occurred while reading {metrics_file_path}: {exc}")
        else:
            print(f"File not found in directory {directory_path}: {metrics_filename}")

    return metrics_records


test_best_metrics = load_metrics_from_directories(matching_directories, "metrics_test_best_1.json")

if test_best_metrics:
    print("Found directories and their corresponding metrics content:")
    for metrics_record in test_best_metrics:
        print(f"Directory path: {metrics_record['directory_path']}")
        print(f"Metrics content: {metrics_record['metrics_data']}")
else:
    print("No directories found containing target metrics files.")


Found directories and their corresponding metrics content:
Directory path: /root/autodl-tmp/em_eqf/checkpoints/rtpp_v2_20260418-175918
Metrics content: {'nll_train_time': -5.559803485870361, 'nll_train_total': -6.810396671295166, 'nll_train_mag': -0.3030048906803131, 'nll_train_b': -0.8905441761016846, 'nll_train_bg': -0.06280841678380966, 'nll_train_bg_norm': -0.2852209806442261, 'nll_val_time': -5.5919718742370605, 'nll_val_total': -6.847859859466553, 'nll_val_mag': -0.2865926921367645, 'nll_val_b': -0.9191149473190308, 'nll_val_bg': -0.038316454738378525, 'nll_val_bg_norm': -0.25090280175209045, 'nll_test_time': -4.856781959533691, 'nll_test_total': -5.998592853546143, 'nll_test_mag': -0.2515993118286133, 'nll_test_b': -0.860139787197113, 'nll_test_bg': -0.014030031859874725, 'nll_test_bg_norm': -0.15035799145698547, 'num_events_train': 22661, 'num_events_val': 4211, 'num_events_test': 3380}
Directory path: /root/autodl-tmp/em_eqf/checkpoints/rtpp_v2_20260418-181941
Metrics content:

In [6]:
target_nll = 0.587
matching_metrics = [
    metrics_record
    for metrics_record in test_best_metrics
    if abs(float(metrics_record["metrics_data"]["nll_test_time"]) - target_nll) < 1e-3
]

print(matching_metrics)


[]


In [7]:
test_metrics = load_metrics_from_directories(matching_directories, "test_metrics.json")
val_metrics = load_metrics_from_directories(matching_directories, "val_metrics.json")


File not found in directory /root/autodl-tmp/em_eqf/checkpoints/rtpp_v2_20260418-175918: test_metrics.json
File not found in directory /root/autodl-tmp/em_eqf/checkpoints/rtpp_v2_20260418-181941: test_metrics.json
File not found in directory /root/autodl-tmp/em_eqf/checkpoints/rtpp_v2_20260420-102834: test_metrics.json
File not found in directory /root/autodl-tmp/em_eqf/checkpoints/rtpp_v2_20260418-175918: val_metrics.json
File not found in directory /root/autodl-tmp/em_eqf/checkpoints/rtpp_v2_20260418-181941: val_metrics.json
File not found in directory /root/autodl-tmp/em_eqf/checkpoints/rtpp_v2_20260420-102834: val_metrics.json


In [8]:
test_metrics

[]

In [9]:
import pandas as pd

rows = []
for metrics_record in val_metrics:
    directory_path = metrics_record["directory_path"]
    metrics_data = metrics_record["metrics_data"]
    row = {"directory_path": directory_path}
    row.update(metrics_data)
    rows.append(row)

metrics_df = pd.DataFrame(rows)
# metrics_df = metrics_df.sort_values(by="directory_path").reset_index(drop=True)

metrics_df = metrics_df[["directory_path", "auc", "f1", "recall", "precision", "R"]]

print(metrics_df)


KeyError: "None of [Index(['directory_path', 'auc', 'f1', 'recall', 'precision', 'R'], dtype='object')] are in the [columns]"

In [ ]:
metrics_df["directory_path"][3]


'/home/yzzhang/pjt/chuandian_eq/checkpoints/rf_9ffe46be'

Clean checkpoints

In [ ]:
import os
import shutil
from datetime import datetime

checkpoints_root_dir = "checkpoints"
delete_before_timestamp = "20250518-205722"
delete_before_datetime = datetime.strptime(delete_before_timestamp, "%Y%m%d-%H%M%S")

for folder_name in os.listdir(checkpoints_root_dir):
    folder_path = os.path.join(checkpoints_root_dir, folder_name)

    if os.path.isdir(folder_path) and folder_name.startswith("classifier_"):
        folder_timestamp = folder_name.split("_")[1]
        folder_datetime = datetime.strptime(folder_timestamp, "%Y%m%d-%H%M%S")

        if folder_datetime < delete_before_datetime:
            print(f"Deleting: {folder_path}")
            shutil.rmtree(folder_path)


ValueError: time data 'se' does not match format '%Y%m%d-%H%M%S'

In [10]:
import src.catalogs
from src.data.catalog import Catalog
print(sorted(Catalog.list_available()))

['AZDX-Base', 'AZDX-Standard', 'Basel-Standard', 'CB_HAB1a-Standard', 'CB_HAB1b-Standard', 'CB_HAB4-Standard', 'ChinaArray-Base', 'ChinaArray-Standard', 'ChuanDian-Base', 'ChuanDian-Standard', 'CooperBasin-Standard', 'ETAS-MultiCatalog', 'ETAS-SingleCatalog', 'FORGE2022-Standard', 'Geysers-Base', 'Geysers-Standard', 'Hauksson-Standard', 'PNR-Base', 'PNR-Standard', 'PNR_1z-Standard', 'PNR_2-Standard', 'QTMSaltonSea-Standard', 'QTMSanJacinto-Standard', 'SCEDC-Standard', 'SSFS-Standard', 'SSFS1993-Standard', 'SSFS2000-Standard', 'SSFS2003-Standard', 'SSFS2004-Standard', 'SSFS2005-Standard', 'St1-2018-Standard', 'St1-2020-Standard', 'White-Standard']
